# 02 フィルタと輪郭 — 物体を数える

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 緑の **✅ チェックポイント** が出たら次へ。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

## 2-0. 準備（題材を撮影 + ヘルパー）
**机の上に小物を数個**置いてカメラに向けると、この後の「物体数え」が楽しくなります。

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
%matplotlib inline
from picamera2 import Picamera2
def show(im, title='', size=(6,4), gray=False):
    plt.figure(figsize=size)
    plt.imshow(im if gray else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap='gray' if gray else None)
    plt.axis('off')
    if title: plt.title(title)
    plt.show()
picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
img = picam.capture_array(); picam.close()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
show(img, '題材')

## 2-1. ぼかし → 二値化（大津）
ノイズをぼかしで抑え、明るさのしきい値で白黒に分けます。**大津の方法**はしきい値を自動で決めます。

In [ ]:
blur = cv2.GaussianBlur(gray, (5,5), 0)
t, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print('大津が選んだしきい値:', t)
show(bw, '二値化（大津）', gray=True)

## 2-2. エッジ検出を**スライダー**で体感
下のセルを実行するとスライダーが出ます。**下しきい値・上しきい値**を動かすと、エッジがリアルタイムに変わります。

In [ ]:
from ipywidgets import interact, IntSlider

@interact(low=IntSlider(80, 0, 255, 5), high=IntSlider(160, 0, 255, 5))
def _(low, high):
    edges = cv2.Canny(blur, low, high)
    plt.figure(figsize=(6,4)); plt.imshow(edges, cmap='gray'); plt.axis('off')
    plt.title(f'Canny(low={low}, high={high})  エッジ画素={int((edges>0).sum())}'); plt.show()

> スライダーが出ない場合は、メニューの **Kernel → Restart** 後にもう一度上から実行してください。

## 2-3. 輪郭抽出で「物体を数える」
二値化画像から輪郭を取り出し、**小さすぎるノイズを面積で捨て**て、残りを数えます。

In [ ]:
from ipywidgets import interact, IntSlider
@interact(min_area=IntSlider(500, 0, 5000, 100))
def _(min_area):
    cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    objs = [c for c in cnts if cv2.contourArea(c) > min_area]
    out = img.copy()
    for c in objs:
        x,y,w,h = cv2.boundingRect(c)
        cv2.rectangle(out, (x,y), (x+w,y+h), (0,255,0), 2)
    cv2.putText(out, f'objects: {len(objs)}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
    plt.figure(figsize=(6,4)); plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'検出 {len(objs)} 個 (min_area={min_area})'); plt.show()

✅ **チェックポイント**：枠で囲まれた物体の数が表示されれば OK。`min_area` を動かして、数え方が変わるのを体感しましょう。

> 数が合わないときは、被写体の背景を白い紙にして明暗の差を上げると安定します。

> 🤖 **ChatGPT に聞いてみよう**
> 「OpenCV の `cv2.threshold` の**大津の方法**って何？ なぜしきい値を自分で決めなくていいの？」と聞いて、仕組みを 3 行で理解しよう。エラーが出たら全文を貼って相談。